# L06 Time-Series Forecasting
This notebook performs data prep, Nixtla forecasting, feature engineering, evaluation, cross-validation, and generative modeling.

In [36]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import mean_absolute_error, mean_squared_error


## Load Dataset (Update path if needed)

In [39]:
df = pd.read_csv('/content/IOT-temp.csv')
df.head()

,id,room_id/id,noted_date,temp,out/in
0,__export__.temp_log_196134_bd201015,Room Admin,08-12-2018 09:30,29,In
1,__export__.temp_log_196131_7bca51bc,Room Admin,08-12-2018 09:30,29,In
2,__export__.temp_log_196127_522915e3,Room Admin,08-12-2018 09:29,41,Out
3,__export__.temp_log_196128_be0919cf,Room Admin,08-12-2018 09:29,41,Out
4,__export__.temp_log_196126_d30b72fb,Room Admin,08-12-2018 09:29,31,In


## Datetime Processing

In [40]:
df['noted_date'] = pd.to_datetime(df['noted_date'], errors='coerce')
df = df.sort_values('noted_date')

## Feature Engineering

In [41]:
df['hour'] = df['noted_date'].dt.hour
df['day'] = df['noted_date'].dt.day
df['month'] = df['noted_date'].dt.month
df['dayofweek'] = df['noted_date'].dt.dayofweek

# Custom features
df['temp_rolling_mean'] = df['temp'].rolling(window=5).mean()
df['temp_diff'] = df['temp'].diff()

df.fillna(method='bfill', inplace=True)
df.head()

/tmp/ipykernel_1022/242504714.py:10: FutureWarning: DataFrame.fillna with 'method' is deprecated and will raise in a future version. Use obj.ffill() or obj.bfill() instead.
  df.fillna(method='bfill', inplace=True)


,id,room_id/id,noted_date,temp,out/in,hour,day,month,dayofweek,temp_rolling_mean,temp_diff
16218,__export__.temp_log_126924_cb744837,Room Admin,2018-01-11 00:06:00,32,In,0.0,11.0,1.0,3.0,38.2,8.0
16217,__export__.temp_log_146101_e61c18d4,Room Admin,2018-01-11 00:07:00,40,Out,0.0,11.0,1.0,3.0,38.2,8.0
16216,__export__.temp_log_111262_7b3ed086,Room Admin,2018-01-11 00:09:00,39,Out,0.0,11.0,1.0,3.0,38.2,-1.0
16215,__export__.temp_log_147650_344507e9,Room Admin,2018-01-11 00:13:00,40,Out,0.0,11.0,1.0,3.0,38.2,1.0
16214,__export__.temp_log_139505_cd77d7f9,Room Admin,2018-01-11 00:23:00,40,Out,0.0,11.0,1.0,3.0,38.2,0.0


## Time-Based Train/Test Split

In [42]:
split_index = int(len(df) * 0.8)
train = df.iloc[:split_index]
test = df.iloc[split_index:]

## Nixtla Forecasting

In [43]:
!pip install nixtla
!pip install tensorflow

In [46]:
from nixtla import NixtlaClient

nixtla_df = df[['noted_date', 'temp']].rename(columns={'noted_date':'ds','temp':'y'})
nixtla_df['unique_id'] = 'temp_series'

# Drop rows where 'ds' (noted_date) is NaT to ensure basic validity
nixtla_df.dropna(subset=['ds'], inplace=True)

# Set 'ds' as index for resampling
nixtla_df.set_index('ds', inplace=True)

# Resample to a fixed frequency (e.g., 'min' for minute) and interpolate
# Aggregate 'y' by mean, and 'unique_id' by first (assuming it's constant)
nixtla_df_resampled = nixtla_df.resample('min').agg({
    'y': 'mean',
    'unique_id': 'first'
})

# Fill NaNs created by resampling in 'y' column, e.g., with ffill and bfill
# This ensures there are no gaps in the time series
nixtla_df_resampled['y'].fillna(method='ffill', inplace=True)
nixtla_df_resampled['y'].fillna(method='bfill', inplace=True)

# Fill NaNs in 'unique_id' (should be 'temp_series' throughout), if any
nixtla_df_resampled['unique_id'].fillna('temp_series', inplace=True)

# Reset index to make 'ds' a column again, which is required by NixtlaClient
nixtla_df = nixtla_df_resampled.reset_index()

client = NixtlaClient(api_key = 'nixak-af4b8619a4e36ebd9d83c8cf172e89b41ce846abcef2ff355190fe99847d497806e04cf47c2b23d0')

forecast = client.forecast(df=nixtla_df, h=24)
forecast.head()

/tmp/ipykernel_1022/256056715.py:21: FutureWarning: A value is trying to be set on a copy of a DataFrame or Series through chained assignment using an inplace method.
The behavior will change in pandas 3.0. This inplace method will never work because the intermediate object on which we are setting values always behaves as a copy.

For example, when doing 'df[col].method(value, inplace=True)', try using 'df.method({col: value}, inplace=True)' or df[col] = df[col].method(value) instead, to perform the operation inplace on the original object.


  nixtla_df_resampled['y'].fillna(method='ffill', inplace=True)
/tmp/ipykernel_1022/256056715.py:21: FutureWarning: Series.fillna with 'method' is deprecated and will raise in a future version. Use obj.ffill() or obj.bfill() instead.
  nixtla_df_resampled['y'].fillna(method='ffill', inplace=True)
/tmp/ipykernel_1022/256056715.py:22: FutureWarning: A value is trying to be set on a copy of a DataFrame or Series through chained assignment using an in

,unique_id,ds,TimeGPT
0,temp_series,2018-12-10 23:56:00,43.716350
1,temp_series,2018-12-10 23:57:00,43.599533
2,temp_series,2018-12-10 23:58:00,43.579155
3,temp_series,2018-12-10 23:59:00,43.549744
4,temp_series,2018-12-11 00:00:00,43.581444


## Evaluation

In [48]:
y_true = test['temp'][:len(forecast)]
y_pred = forecast['TimeGPT']

mae = mean_absolute_error(y_true, y_pred)
mse = mean_squared_error(y_true, y_pred)

print("MAE:", mae)
print("MSE:", mse)

MAE: 4.791620375
MSE: 22.982767516498626


## Rolling Cross-Validation

In [49]:
window_size = int(len(df)*0.6)
errors = []

for i in range(window_size, len(df)-24):
    train_cv = df.iloc[:i]
    test_cv = df.iloc[i:i+24]
    pred = [train_cv['temp'].iloc[-1]] * len(test_cv)
    error = mean_absolute_error(test_cv['temp'], pred)
    errors.append(error)

print("Average CV MAE:", np.mean(errors))

Average CV MAE: 1.3320668563178624


## Generative Modeling (Autoencoder)

In [50]:
from tensorflow.keras import layers, models

temp_data = df['temp'].values.reshape(-1,1)

model = models.Sequential([
    layers.Dense(16, activation='relu', input_shape=(1,)),
    layers.Dense(8, activation='relu'),
    layers.Dense(1)
])

model.compile(optimizer='adam', loss='mse')
model.fit(temp_data, temp_data, epochs=10, batch_size=32)

synthetic = model.predict(temp_data)


Epoch 1/10


/usr/local/lib/python3.12/dist-packages/keras/src/layers/core/dense.py:106: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(activity_regularizer=activity_regularizer, **kwargs)


3051/3051 ━━━━━━━━━━━━━━━━━━━━ 5s 1ms/step - loss: 29.2589 
Epoch 2/10
3051/3051 ━━━━━━━━━━━━━━━━━━━━ 6s 2ms/step - loss: 0.0073
Epoch 3/10
3051/3051 ━━━━━━━━━━━━━━━━━━━━ 5s 2ms/step - loss: 9.3040e-04
Epoch 4/10
3051/3051 ━━━━━━━━━━━━━━━━━━━━ 5s 1ms/step - loss: 8.5567e-07
Epoch 5/10
3051/3051 ━━━━━━━━━━━━━━━━━━━━ 6s 2ms/step - loss: 4.3928e-05
Epoch 6/10
3051/3051 ━━━━━━━━━━━━━━━━━━━━ 5s 2ms/step - loss: 5.9026e-05
Epoch 7/10
3051/3051 ━━━━━━━━━━━━━━━━━━━━ 5s 2ms/step - loss: 7.9803e-05
Epoch 8/10
3051/3051 ━━━━━━━━━━━━━━━━━━━━ 6s 2ms/step - loss: 5.9213e-05
Epoch 9/10
3051/3051 ━━━━━━━━━━━━━━━━━━━━ 5s 2ms/step - loss: 6.1884e-05
Epoch 10/10
3051/3051 ━━━━━━━━━━━━━━━━━━━━ 6s 2ms/step - loss: 5.3004e-05
3051/3051 ━━━━━━━━━━━━━━━━━━━━ 3s 1ms/step
